**At the start of each notebook there will be 1 - 3 cells of code that need to be run before the rest of the notebook. These cells make sure we have imported all the required Python packages so that we can run the rest of the code without issue!**
- If you do not have these packages, you will need to pip install them. 

In [ ]:
import numpy as np # Numpy is used for math and stats in python 
import pandas as pd # Pandas is used for handling data in a spreadsheet like way
import matplotlib.pyplot as plt  # Matplotlib is used for plotting

In [2]:
## Matt has made a lot of improvements over the last week as we put together this workshop - run this upgrade! Then restart your kernel
#%pip install --upgrade petthermotools

In [3]:
# If the alphaMELTS for Python files have not been added to your Python path (see installation guide) then use the two lines below to add
# the location of the alphaMELTS files here.

# import sys
# sys.path.append(r'C:\Users\penny\Box\Berkeley_new\MELTS_Installation\alphamelts_py\alphamelts-py-2.3.1-win64')


import petthermotools as ptt 

print(ptt.__version__)

alphaMELTS for Python files successfully located.
If using the Green et al. (2025) or Weller et al. (2024) thermodynamic models please run `ptt.activate_petthermotools_env()` prior to any calculations.
1.3.1


## Lets quickly talk through datatypes
- String are letters and words like  Model = "MELTSv1.2.0" or Model = "pMELTS"
- Dataframes can be thought of like an excel sheet - there are columns, with data in them. This data can be a mix of integers (like 3), floats (other numbers, like 3.0, 3.1), and strings (like Volcano Names!)
- Pandas Series are single columns of a dataframe, like one column in an excel spreadsheet
- Dictionaries are sets of data. In PetThermoTools, they are used to return multiple dataframes from a function, like a dataframe of Cpx comps, a dataframe of liquid comps, a dataframe of thermodynamic properties etc. 

# Part 1 - Python basics

First, to get used to data handling in Python, let's load in some data and practice! The following dataset is a compilation of whole-rock data from the Aleutian volcanic arc. We'll use Pandas to import this spreadsheet as a Pandas DataFrame:

In [4]:
Aleu = pd.read_excel("Aleutians.xlsx")
Aleu.head()

,Volcano,SiO2_Liq,TiO2_Liq,Al2O3_Liq,FeOt_Liq,MnO_Liq,MgO_Liq,CaO_Liq,Na2O_Liq,K2O_Liq,P2O5_Liq
0,Augustine,51.821911,0.584089,15.971940,8.038338,0.159048,7.812462,11.044827,2.202869,0.348145,0.110600
1,Augustine,51.821911,0.584089,15.971940,8.038338,0.159048,7.812462,11.044827,2.202869,0.348145,0.110600
2,Augustine,54.254823,0.586529,15.396650,7.798657,0.169585,8.079017,10.791099,2.425790,0.382063,0.115787
3,Augustine,54.254823,0.586529,15.396650,7.798657,0.169585,8.079017,10.791099,2.425790,0.382063,0.115787
4,Augustine,54.038554,0.591405,15.028963,7.897825,0.170651,8.273892,11.172488,2.343959,0.371325,0.110927


You should see that there is a column header called "Volcano". We can investigate which volcanoes we have data for by using the `.unique()` method. 

**Please follow along and fill in the next 5 code cells as I talk through each one**.

We can also use the column headers to isolate any of the chemical parameters in a pandas series:

and we can also use pandas to filter the dataset by volcano, for example we can examine which rows contain "Okmok" in the 'Volcano' column:

and then use this to filter the main dataset:

If we wanted, we could save this filtered dataset as a new variable:

Finally, we can use `matplotlib` alongside  `pandas` to plot up the chemical data from the Aleutian datasets. For example, plotting the data for a single volcano:

In [ ]:
plt.plot()
plt.xlabel('MgO (wt%)')
plt.ylabel('FeO$_t$ (wt%)')

Or we could use `subplots` alongside a `for` loop to plot multiple oxides for all volcanoes in the dataset:

In [ ]:
f, a = plt.subplots(1,2, figsize = (8,3.5))
# for volc in Aleu['Volcano'].unique():

a[0].set_xlabel('MgO (wt%)')
a[1].set_xlabel('MgO (wt%)')

a[0].set_ylabel('FeO$_t$ (wt%)')
a[1].set_ylabel('CaO (wt%)')

a[0].legend()

f.tight_layout()

# Part 2 - Basic calculations

## Finding the liquidus
Time to move onto some MELTS calculations! Many functions in `petthermotools` can use compositions from dataframes, such as that above, as the starting point for thermodynamic calculations. One potentially useful place to start is with finding the liquidus temperature of each sample.

Using the `Okmok` dataframe we created above use the code cell below to calculate the liquidus temperature at a pressure of 2000 bar, test using different water contents to see how these influence the calculated liquidus temperatures if you are waiting for others to finish.

In [ ]:
Okmok.head()

In [ ]:
liquidus_T = ptt.findLiq_multi(bulk=Okmok,
                               Model="MELTSv1.0.2",
                               P_bar = 2000,
                               fO2_buffer="NNO",
                               fO2_offset=1,
                               H2O_Liq = 2)

In [ ]:
liquidus_T.head()

## Investigating phase stability

Now that we have a liquidus temperature estimate for each sample we an use another `petthermotools` function to investigate phase stability below the liquidus. `equilibrate_multi` allows us to pass multiple compositions to petthermotools at user specified P-T-H2O-fO2 conditions and calculate the thermodynamic equilibrium. Using the code below equilibriate the Okmok samples at 5, 20, and 50 $^o$C below the liquidus and examine the influence on phase stability:

**I've deliberately left some of the arguments blank here so you'll have to enter values for each**.

In [ ]:
equil = ptt.equilibrate_multi(Model="MELTSv1.0.2",
                              bulk = ,
                              T_C = liquidus_T['T_Liq_C'].values - 20,
                              P_bar = ,
                              H2O_init=,
                              fO2_buffer=,
                              fO2_offset=)

In [ ]:
equil['PhaseList']

These equilibrate calculations share a similar output structure to many of the petthermotools calculations, so let's spend some time examining that structure:

# Part 3 - Crystallization Models

Time to move onto some more complex models! One of the most common use of the MELTS models is to simulate magma differentiation, primarily through fractional crystallization. We can use the `help` function to examine the petthermotools `isobaric_crysatllisation` function:

In [17]:
help(ptt.isobaric_crystallisation)

Help on function isobaric_crystallisation in module petthermotools.Path_wrappers:

isobaric_crystallisation(Model=None, bulk=None, Frac_solid=None, Frac_fluid=None, T_path_C=None, T_start_C=None, T_end_C=None, dt_C=None, P_bar=None, Fe3Fet_init=None, Fe3Fet_Liq=None, H2O_init=None, CO2_init=None, H2O_Liq=None, CO2_Liq=None, find_liquidus=False, fO2_buffer=None, fO2_offset=None, label=None, Crystallinity_limit=None, fluid_sat=False, timeout=None, Suppress=['rutile', 'tridymite'], Suppress_except=False, multi_processing=True, quiet_engine=True, MgO_limit=None)
    Simulates isobaric (constant-pressure) crystallization paths using the `multi_path` function.
    
    Parameters
    ----------
    Model : str, optional
        Thermodynamic model, options:
            MELTS:
                "pMELTS"
                "MELTSv1.0.2"
                "MELTSv1.1.0"
                "MELTSv1.2.0"
            MAGEMin:
                "Green2025"
                "Weller2024"
    bulk : dict or pandas.

Before we run any calculations let's first isolate a single starting composition. One possibility is to take the Okmok composition with the highest MgO content as our starting point:

In [ ]:
starting_comp = Okmok.loc[Okmok['MgO_Liq'] == np.nanmax(Okmok['MgO_Liq'])].squeeze().to_dict()
starting_comp

In [ ]:
frac_xtal = ptt.isobaric_crystallisation(Model="MELTSv1.0.2",
                                         bulk = starting_comp,
                                         P_bar = np.array([1000,2000,4000,6000]),
                                         find_liquidus=True,
                                         dt_C=2,
                                         T_end_C=600,
                                         Frac_solid=True,
                                         Frac_fluid=True,
                                         fO2_buffer="NNO",
                                         fO2_offset=1,
                                         H2O_init = 2,
                                         label = "P_bar")

Let's examine the output structure again:

Within petthermotools we've also created several in-built functions for visualizing thermodynamic results. Use the cells below to plot the liquid-line-of-descent using the `harker` function and the phase proportions using the `phase_plot` function:

In [ ]:
ptt.harker(Results = frac_xtal, 
            data = {"Augustine": Aleu.loc[Aleu['Volcano'] == "Augustine"],
                    "Okmok": Aleu.loc[Aleu['Volcano'] == "Okmok"],
                    "Shishaldin": Aleu.loc[Aleu['Volcano'] == "Shishaldin"],})

Use `help(ptt.phase_plot)` to understand the input structure for this function:

In [ ]:
ptt.phase_plot()

Finally we can compare the abundance of different phases between models using the `ptt.phase_mass_comparison` function:

In [ ]:
ptt.phase_mass_comparison(Results=frac_xtal, x_axis = "MgO_Liq", phase = "plagioclase1", cmap = "Reds")

# Part 4 - Do some calculations in your own time

Repeat the calculations above, changing the initial water content and/or fO2 instead of pressure. How do these variables influence phase stability. With these melts calculations can you provide any insights into the role of magmatic conditions on the difference between calc-alkaline and tholeiitic differentition trends? Perhaps try plotting SiO2 against FeO/MgO (e.g., https://www.researchgate.net/figure/FeO-MgO-vs-SiO-2-diagram-showing-differences-between-calc-alkaline-Powder-River-Volcanic_fig2_291970605).

In [ ]:
frac_xtal_fO2 = ptt.isobaric_crystallisation(Model="MELTSv1.0.2",
                                         bulk = starting_comp,)

In [ ]:
ptt.harker(Results = frac_xtal_fO2, 
            data = {"Augustine": Aleu.loc[Aleu['Volcano'] == "Augustine"],
                    "Okmok": Aleu.loc[Aleu['Volcano'] == "Okmok"],
                    "Shishaldin": Aleu.loc[Aleu['Volcano'] == "Shishaldin"],})